# GNN Training - Overnight Micro-Batch Training

Features:
- Micro-batches to avoid OOM
- Checkpoint saving every N epochs
- Resume from checkpoint if interrupted
- Can run overnight

Output: graph_encoder.pt

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

OUTPUT_DIR = "/content/drive/MyDrive/EVA_Ai_Exports"
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "gnn_checkpoints")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

MODEL_PATH = os.path.join(OUTPUT_DIR, "graph_encoder.pt")
print(f"Output: {OUTPUT_DIR}")
print(f"Checkpoints: {CHECKPOINT_DIR}")

In [ ]:
!pip install -q torch numpy

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import json
import os
import time
from datetime import datetime

print(f"PyTorch version: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
# Generate synthetic training data (micro-batches)
np.random.seed(42)
torch.manual_seed(42)

# Graph parameters
NUM_NODES = 1000
INPUT_DIM = 384
HIDDEN_DIM = 512
OUTPUT_DIM = 2560

# Generate node embeddings
node_embeddings = np.random.randn(NUM_NODES, INPUT_DIM).astype(np.float32)
node_embeddings = node_embeddings / (np.linalg.norm(node_embeddings, axis=1, keepdims=True) + 1e-8)

# Generate edges (sparse)
NUM_EDGES = 5000
edge_index = np.random.randint(0, NUM_NODES, (2, NUM_EDGES))

# Target graph vectors (what GNN should learn)
target_vectors = np.random.randn(NUM_NODES, OUTPUT_DIM).astype(np.float32)
target_vectors = target_vectors / (np.linalg.norm(target_vectors, axis=1, keepdims=True) + 1e-8)

print(f"Generated {NUM_NODES} nodes, {NUM_EDGES} edges")
print(f"Node embeddings shape: {node_embeddings.shape}")
print(f"Target vectors shape: {target_vectors.shape}")

In [ ]:
# Simple GNN model (SAGEConv-like)
class SimpleGNN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.lin1 = nn.Linear(input_dim, hidden_dim)
        self.lin2 = nn.Linear(hidden_dim, hidden_dim)
        self.lin3 = nn.Linear(hidden_dim, output_dim)
        self.proj = nn.Linear(output_dim, output_dim)
    
    def forward(self, x, edge_index):
        # Simple graph convolution via neighbor aggregation
        h = F.relu(self.lin1(x))
        
        # Aggregate neighbors (simplified)
        h = self._aggregate(h, edge_index)
        h = F.relu(self.lin2(h))
        h = self._aggregate(h, edge_index)
        h = self.lin3(h)
        
        # Global pooling
        graph_vec = h.mean(dim=0, keepdim=True)
        graph_vec = self.proj(graph_vec)
        
        return graph_vec
    
    def _aggregate(self, h, edge_index):
        # Simple mean aggregation
        src, dst = edge_index
        aggregated = torch.zeros_like(h)
        for i in range(h.shape[0]):
            mask = (src == i) | (dst == i)
            neighbors = h[mask]
            if len(neighbors) > 0:
                aggregated[i] = neighbors.mean(dim=0)
        return aggregated

print("SimpleGNN defined")

In [ ]:
# Optimized GNN with batch processing
class OptimizedGNN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.lin1 = nn.Linear(input_dim, hidden_dim)
        self.lin2 = nn.Linear(hidden_dim, hidden_dim)
        self.lin3 = nn.Linear(hidden_dim, output_dim)
        self.proj = nn.Linear(output_dim, output_dim)
        self.dropout = nn.Dropout(0.1)
    
    def forward(self, x, edge_index):
        # x: [num_nodes, input_dim]
        # edge_index: [2, num_edges]
        
        # Layer 1
        h = F.relu(self.lin1(x))
        h = self.dropout(h)
        
        # Message passing (simplified SAGEConv)
        h = self._sage_layer(h, edge_index)
        h = F.relu(h)
        
        # Layer 2
        h = self._sage_layer(h, edge_index)
        h = F.relu(h)
        
        # Layer 3
        h = self.lin3(h)
        
        # Global mean pooling
        graph_vec = h.mean(dim=0, keepdim=True)
        graph_vec = self.proj(graph_vec)
        
        return graph_vec
    
    def _sage_layer(self, h, edge_index):
        src, dst = edge_index
        num_nodes = h.shape[0]
        
        # Aggregate: sum messages from neighbors
        msg = torch.zeros_like(h)
        msg.index_add_(0, dst, h[src])
        
        # Count neighbors per node
        count = torch.zeros(num_nodes, device=h.device)
        count.index_add_(0, dst, torch.ones_like(dst, dtype=torch.float))
        count = count.clamp(min=1)
        
        # Normalize
        msg = msg / count.unsqueeze(-1)
        
        return msg + h  # Residual connection

model = OptimizedGNN(INPUT_DIM, HIDDEN_DIM, OUTPUT_DIM).to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Convert data to tensors
node_tensor = torch.from_numpy(node_embeddings).float().to(device)
edge_tensor = torch.from_numpy(edge_index).long().to(device)
target_tensor = torch.from_numpy(target_vectors).float().to(device)

print(f"Data ready on {device}")

In [ ]:
# Training config
EPOCHS = 1000
LEARNING_RATE = 1e-3
BATCH_SIZE = 100  # Micro-batches
CHECKPOINT_EVERY = 50  # Save every N epochs
NUM_BATCHES = NUM_NODES // BATCH_SIZE

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
criterion = nn.MSELoss()

print(f"Training config:")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Batches per epoch: {NUM_BATCHES}")
print(f"  Checkpoint every: {CHECKPOINT_EVERY} epochs")

In [ ]:
# Check for existing checkpoint
def find_latest_checkpoint():
    checkpoints = [f for f in os.listdir(CHECKPOINT_DIR) if f.startswith('checkpoint_epoch_')]
    if not checkpoints:
        return None
    # Sort by epoch number
    checkpoints.sort(key=lambda x: int(x.split('_')[-1].split('.')[0]))
    return os.path.join(CHECKPOINT_DIR, checkpoints[-1])

latest_checkpoint = find_latest_checkpoint()
start_epoch = 0

if latest_checkpoint:
    print(f"Found checkpoint: {latest_checkpoint}")
    checkpoint = torch.load(latest_checkpoint)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch']
    print(f"Resuming from epoch {start_epoch}")
else:
    print("Starting from scratch")

In [ ]:
# Training loop with micro-batches
print(f"Starting training from epoch {start_epoch}...")
print(f"Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

model.train()
total_start_time = time.time()

for epoch in range(start_epoch, EPOCHS):
    epoch_start = time.time()
    total_loss = 0
    
    # Shuffle indices for each epoch
    indices = torch.randperm(NUM_NODES)
    
    for batch_idx in range(NUM_BATCHES):
        start_idx = batch_idx * BATCH_SIZE
        end_idx = min(start_idx + BATCH_SIZE, NUM_NODES)
        batch_indices = indices[start_idx:end_idx]
        
        # Get batch data
        batch_nodes = node_tensor[batch_indices]
        batch_targets = target_tensor[batch_indices]
        
        # Forward pass
        graph_vec = model(batch_nodes, edge_tensor)
        
        # Loss: predict target for batch
        target = batch_targets.mean(dim=0, keepdim=True)
        loss = criterion(graph_vec, target)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    avg_loss = total_loss / NUM_BATCHES
    epoch_time = time.time() - epoch_start
    
    # Log every 10 epochs
    if (epoch + 1) % 10 == 0:
        elapsed = time.time() - total_start_time
        print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.6f} | Time: {epoch_time:.2f}s | Total: {elapsed/60:.1f}min")
    
    # Save checkpoint
    if (epoch + 1) % CHECKPOINT_EVERY == 0:
        checkpoint_path = os.path.join(CHECKPOINT_DIR, f"checkpoint_epoch_{epoch+1}.pt")
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': avg_loss
        }, checkpoint_path)
        print(f"Checkpoint saved: {checkpoint_path}")

total_time = time.time() - total_start_time
print(f"\nTraining complete!")
print(f"Total time: {total_time/60:.1f} minutes")

In [ ]:
# Save final model
print(f"Saving model to {MODEL_PATH}...")
torch.save(model.state_dict(), MODEL_PATH)
print("Model saved!")
!ls -lh $MODEL_PATH

In [ ]:
# Download final model
from google.colab import files
files.download(MODEL_PATH)
print("Download started!")